In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Optional models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# -----------------------
# PAGE CONFIG
# -----------------------
st.set_page_config(page_title="Smart Construction AI", layout="wide")

st.title("🏗️ Smart Construction AI Dashboard")
st.markdown("BIM + IoT Simulation + Machine Learning")

# -----------------------
# SIDEBAR
# -----------------------
st.sidebar.header("⚙️ Controls")

days = st.sidebar.slider("Simulation Days", 50, 200, 120)
live_sim = st.sidebar.checkbox("📡 Live IoT Simulation", True)

# -----------------------
# IoT DATA GENERATOR
# -----------------------
def generate_data(base, days):
    data = pd.DataFrame()
    data["Labour"] = np.random.randint(20, 100, days)
    data["Rain"] = np.random.randint(0, 5, days)
    data["Temp"] = np.random.randint(25, 40, days)
    data["Truck"] = np.random.randint(2, 10, days)
    data["Quality"] = np.random.randint(1, 4, days)

    data["Usage"] = (
        base
        + data["Labour"] * 0.6
        - data["Rain"] * 3
        + data["Temp"] * 0.4
        - data["Truck"] * 1.2
        + data["Quality"] * 6
        + np.random.normal(0, 5, days)
    )
    return data

# -----------------------
# UPLOAD REVIT DATA
# -----------------------
st.subheader("📂 Upload Revit Data")

file = st.file_uploader("Upload CSV", type=["csv"])

if file:
    revit = pd.read_csv(file)
    st.dataframe(revit.head())

    base_value = revit.iloc[0, 0]

    # -----------------------
    # GENERATE IoT DATA
    # -----------------------
    data = generate_data(base_value, days)

    # -----------------------
    # LIVE SIMULATION
    # -----------------------
    st.subheader("📡 Live IoT Simulation")

    chart = st.empty()

    if live_sim:
        for i in range(10):
            temp_data = data.head(i+10)
            chart.line_chart(temp_data["Usage"])
            time.sleep(0.3)

    else:
        st.line_chart(data["Usage"])

    # -----------------------
    # MODEL TRAINING
    # -----------------------
    st.subheader("🤖 Model Comparison")

    X = data.drop("Usage", axis=1)
    y = data["Usage"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    models = {
        "Random Forest": RandomForestRegressor(),
        "XGBoost": XGBRegressor(),
        "LightGBM": LGBMRegressor()
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        error = mean_absolute_error(y_test, pred)
        results[name] = error

    # Show results
    st.write("### 📊 Model Performance (MAE)")

    result_df = pd.DataFrame({
        "Model": list(results.keys()),
        "Error": list(results.values())
    })

    st.dataframe(result_df)
    st.bar_chart(result_df.set_index("Model"))

    best_model_name = min(results, key=results.get)
    st.success(f"🏆 Best Model: {best_model_name}")

    best_model = models[best_model_name]

    # -----------------------
    # KPI CARDS
    # -----------------------
    st.subheader("📊 Key Metrics")

    col1, col2, col3 = st.columns(3)

    col1.metric("Avg Usage", f"{data['Usage'].mean():.2f}")
    col2.metric("Max Usage", f"{data['Usage'].max():.2f}")
    col3.metric("Min Usage", f"{data['Usage'].min():.2f}")

    # -----------------------
    # PREDICTION INPUT
    # -----------------------
    st.subheader("📊 Predict Usage")

    c1, c2, c3 = st.columns(3)

    labour = c1.slider("Labour", 10, 100, 50)
    rain = c1.slider("Rain Delay", 0, 5, 2)

    temp = c2.slider("Temperature", 25, 40, 30)
    truck = c2.slider("Truck Time", 2, 10, 5)

    quality = c3.selectbox("Quality", [1, 2, 3])

    input_data = np.array([[labour, rain, temp, truck, quality]])
    pred = best_model.predict(input_data)

    st.metric("Predicted Usage", f"{pred[0]:.2f}")

    # -----------------------
    # 7 DAY FORECAST
    # -----------------------
    st.subheader("📅 7-Day Forecast")

    future = []

    for i in range(7):
        rand_input = np.array([[
            np.random.randint(20, 100),
            np.random.randint(0, 5),
            np.random.randint(25, 40),
            np.random.randint(2, 10),
            np.random.randint(1, 4)
        ]])
        future.append(best_model.predict(rand_input)[0])

    forecast_df = pd.DataFrame({"Day": range(1, 8), "Usage": future})

    st.line_chart(forecast_df.set_index("Day"))

else:
    st.info("👆 Upload Revit data to start")

# -----------------------
# FOOTER
# -----------------------
st.markdown("---")
st.markdown("🚀 Developed by Piyush & Team | Smart Construction AI")